---

## **DIPLOME UNIVERSITAIRE SDA**


## **Projet Generative AI — Système Agentique d'Évaluation et d'Anticipation des Risques Climatiques**





Promotion 007

Avril 2026




**Corpus** : 10 rapports scientifiques (GIEC AR6, Copernicus, EM-DAT, NOAA, JRC, WMO, EU CELEX)

**Repo** : https://github.com/diegomerchanm/catastrophes-climatiques-rag

---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans recalculer si les fichiers existent déjà)
- **traçable** (quality gates go/no-go explicites)

**Convention :** chaque cellule de code doit produire une sortie visible. Si aucune sortie naturelle, ajouter un `print()` de vérification.

---

---

## Contexte du projet

**Architecture :**
```
Question utilisateur
    ↓
Agent Agentic RAG (ReAct — LangGraph)
    ↓ décide seul quels outils appeler
    ├── search_corpus (RAG hybride BM25 + Dense + reranking)
    ├── get_weather / get_historical_weather / get_forecast (OpenMeteo)
    ├── web_search (Tavily + DuckDuckGo fallback)
    ├── calculator
    └── send_email
    ↓ boucle Reason → Act → Observe → Repeat
Réponse argumentée avec sources
```

**Stack :** LangChain + LangGraph + Anthropic Claude (Haiku/Sonnet/Opus) + FAISS + Chainlit + MLflow

---

# NOTEBOOK 8 — LLMOps / Monitoring

**Auteur :** Xia Bizot

---

### Objectif

Démontrer le suivi opérationnel du système en conditions réelles : monitoring des tokens, estimation des coûts, spécialisation des modèles, versioning des prompts, fallback LLM, et distinction LLMOps vs MLOps vs DevOps.

---

### Plan du notebook

| Section | Contenu |
|---|---|
| 1. Configuration | Imports, chemins, seed, versions, constantes |
| 2. LLMOps vs MLOps vs DevOps | Distinction et positionnement du projet |
| 3. Monitoring des tokens | Suivi par question, par agent, cumulatif |
| 4. Estimation des coûts | Par question, par agent, projections |
| 5. Spécialisation LLM | Haiku vs Sonnet vs Opus, justification |
| 6. Versioning des prompts | v1.0 vs v2.0, impact sur les réponses |
| 7. Fallback LLM | Sonnet → Haiku → Ollama |
| 8. Dashboard MLflow | Visualisation des runs |
| 9. Résultats | Tableaux, visualisations |
| 10. Conclusions | Synthèse, quality gate, décisions |

---

### Hypothèse testable

> La spécialisation des modèles (Haiku pour les tâches simples, Sonnet pour le RAG) réduit le coût total d'au moins 50% par rapport à un usage Sonnet uniforme, sans dégradation significative de la qualité.

---

---

## 1. Configuration

### 1.1. Imports et timing

In [ ]:
import os
import sys
import time
import logging
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

notebook_start_time = time.time()

print('>> 1.1. Imports : OK')

### 1.2. Chemins et environnement

In [ ]:
BASE = Path('..')
OUTPUT_DIR = BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE))

from dotenv import load_dotenv
load_dotenv(BASE / '.env')

print(f'Dossier output : {OUTPUT_DIR.resolve()}')
print('>> 1.2. Chemins : OK')

### 1.3. Versions et seed

In [ ]:
SEED = 42
np.random.seed(SEED)

print(f'Python     : {sys.version}')
print(f'Pandas     : {pd.__version__}')
print(f'Numpy      : {np.__version__}')
print(f'SEED       : {SEED}')
print('>> 1.3. Versions / seed : OK')

### 1.4. Constantes du projet

In [ ]:
from src.config import (
    AGENT_CONFIGS, TOKEN_TRACKING,
    MODEL_HAIKU, MODEL_SONNET, MODEL_OPUS,
    MODEL_PRICING, TokenCounter,
)
from src.prompts.agent_prompts import PROMPTS, CURRENT_VERSION, list_versions

print(f'TOKEN_TRACKING     : {TOKEN_TRACKING}')
print(f'Prompt version     : {CURRENT_VERSION}')
print(f'Versions dispo     : {list_versions()}')
print(f'Nb agents config   : {len(AGENT_CONFIGS)}')
print()
for agent, cfg in AGENT_CONFIGS.items():
    print(f'  {agent:15s} → {cfg["model"]:40s} temp={cfg["temperature"]} max_tokens={cfg["max_tokens"]}')
print('>> 1.4. Constantes projet : OK')

### 1.5. Quality gates

In [ ]:
checks = {
    'cle_anthropic': (bool(os.getenv('ANTHROPIC_API_KEY')), bool(os.getenv('ANTHROPIC_API_KEY'))),
    'token_tracking_actif': (TOKEN_TRACKING, TOKEN_TRACKING),
    'nb_agents_configures': (len(AGENT_CONFIGS), len(AGENT_CONFIGS) == 6),
    'prompts_versions': (len(PROMPTS), len(PROMPTS) >= 2),
}

all_ok = True
for k, (valeur, condition) in checks.items():
    status = '[OK]' if condition else '[KO]'
    if not condition:
        all_ok = False
    print(f'  {status} {k} : {valeur}')

assert all_ok, 'Quality gates KO'
print('>> 1.5. Quality gates : OK')

---

## 2. LLMOps vs MLOps vs DevOps

### Distinction

| | DevOps | MLOps | LLMOps |
|---|---|---|---|
| **Objet** | Code applicatif | Modèle ML entraîné | LLM externe (API) |
| **Versioning** | Code (Git) | Code + modèle + données | Code + **prompts** |
| **Entraînement** | Aucun | Train/Eval/Deploy | Aucun (pré-entraîné) |
| **Monitoring** | Uptime, latence | Drift, métriques ML | **Tokens, coûts, faithfulness** |
| **Reproductibilité** | Docker, CI/CD | + seeds, splits | + **version du prompt** |
| **Fallback** | Load balancer | Modèle de secours | **LLM de secours** (Haiku, Ollama) |

### Ce qu'on applique dans ce projet

- **DevOps** : Git, Docker, CI/CD GitHub Actions, Azure
- **MLOps** : MLflow tracking, tests pytest, pipeline reproductible
- **LLMOps** : monitoring tokens/coûts, prompt versioning, fallback triple, spécialisation LLM

---

---

## 3. Monitoring des tokens

Lancer une série de questions et observer la consommation de tokens par question et par agent.

In [ ]:
from src.agents.agent import run_agent, get_token_summary, reset_token_counter

QUESTIONS_TEST = [
    'Bonjour, je suis Kamila',
    'Quelles catastrophes en Méditerranée en 2023 ?',
    'Quel temps fait-il à Marseille ?',
    'Combien font 3+7*2 ?',
    'Quel risque d\'inondation à Lyon cette semaine ?',
    'Comment je m\'appelle ?',
]

results_tokens = []

for i, question in enumerate(QUESTIONS_TEST):
    reset_token_counter()
    t0 = time.time()
    reponse = run_agent(question)
    duree = round(time.time() - t0, 2)
    tokens = get_token_summary()
    
    results_tokens.append({
        'question': question[:50],
        'tokens_in': tokens['total_input_tokens'],
        'tokens_out': tokens['total_output_tokens'],
        'total_tokens': tokens['total_tokens'],
        'cout_usd': tokens['estimated_cost_usd'],
        'duree_s': duree,
        'nb_outils': sum(tokens['calls_by_agent'].values()),
    })
    print(f'  [{i+1}/{len(QUESTIONS_TEST)}] {question[:40]:40s} → {tokens["total_tokens"]:6d} tokens, ${tokens["estimated_cost_usd"]:.4f}, {duree}s')

df_tokens = pd.DataFrame(results_tokens)
print(f'\nTotal tokens session : {df_tokens["total_tokens"].sum()}')
print(f'Coût total session  : ${df_tokens["cout_usd"].sum():.4f}')
print('>> 3. Monitoring tokens : OK')

### Analyse

[Interpréter les résultats : quelles questions consomment le plus ? pourquoi ?]

---

---

## 4. Estimation des coûts

In [ ]:
# Tableau des coûts par question
print(df_tokens[['question', 'tokens_in', 'tokens_out', 'total_tokens', 'cout_usd', 'duree_s']].to_string())

# Projection
cout_moyen = df_tokens['cout_usd'].mean()
print(f'\nCoût moyen par question : ${cout_moyen:.4f}')
print(f'Projection 100 questions  : ${cout_moyen * 100:.2f}')
print(f'Projection 1000 questions : ${cout_moyen * 1000:.2f}')
print(f'Projection 10000 questions: ${cout_moyen * 10000:.2f}')

# Comparatif : si tout en Opus
tokens_total = df_tokens['total_tokens'].sum()
cout_opus = (tokens_total / 1_000_000) * (MODEL_PRICING[MODEL_OPUS]['input'] + MODEL_PRICING[MODEL_OPUS]['output']) / 2
cout_haiku = (tokens_total / 1_000_000) * (MODEL_PRICING[MODEL_HAIKU]['input'] + MODEL_PRICING[MODEL_HAIKU]['output']) / 2
cout_reel = df_tokens['cout_usd'].sum()

print(f'\nComparatif coût pour {tokens_total} tokens :')
print(f'  Tout Haiku  : ${cout_haiku:.4f}')
print(f'  Spécialisé  : ${cout_reel:.4f} (notre approche)')
print(f'  Tout Opus   : ${cout_opus:.4f}')
print(f'  Économie vs Opus : {(1 - cout_reel/cout_opus)*100:.0f}%' if cout_opus > 0 else '')
print('>> 4. Estimation coûts : OK')

In [ ]:
# Visualisation coûts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tokens par question
df_tokens.plot(x='question', y='total_tokens', kind='barh', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('Tokens consommés par question')
axes[0].set_xlabel('Tokens')

# Coût par question
df_tokens.plot(x='question', y='cout_usd', kind='barh', ax=axes[1], color='#e74c3c', legend=False)
axes[1].set_title('Coût estimé par question ($)')
axes[1].set_xlabel('USD')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'NB8_tokens_couts.png', dpi=200, bbox_inches='tight')
print(f'  [OK] NB8_tokens_couts.png sauvegardé')
plt.show()

### Analyse

[Interpréter : questions RAG vs chat simple en termes de coût. Justifier la spécialisation.]

---

---

## 5. Spécialisation LLM

Montrer que chaque agent utilise le modèle adapté à sa complexité.

In [ ]:
# Tableau de spécialisation
spec_data = []
for agent, cfg in AGENT_CONFIGS.items():
    modele = cfg['model']
    pricing = MODEL_PRICING.get(modele, {})
    spec_data.append({
        'agent': agent,
        'modèle': modele.split('-')[1] if '-' in modele else modele,
        'temperature': cfg['temperature'],
        'max_tokens': cfg['max_tokens'],
        'prix_input_M': pricing.get('input', 0),
        'prix_output_M': pricing.get('output', 0),
        'description': cfg['description'],
    })

df_spec = pd.DataFrame(spec_data)
print(df_spec.to_string())
print('>> 5. Spécialisation LLM : OK')

### Analyse

**Justification de la spécialisation :**
- **Orchestrateur (Sonnet)** : doit raisonner et choisir les bons outils → qualité requise
- **RAG (Sonnet)** : synthèse de documents longs → qualité requise
- **Météo/Web/Chat (Haiku)** : tâches factuelles, pas de raisonnement complexe → économie
- **Analyste (Opus)** : croisement multi-sources, analyse de risque → capacité max

---

---

## 6. Versioning des prompts

In [ ]:
from src.prompts.agent_prompts import get_prompt

for version in list_versions():
    prompt = get_prompt(version)
    print(f'=== Prompt {version} ({len(prompt)} caractères) ===')
    print(prompt[:200] + '...')
    print()

print('>> 6. Versioning prompts : OK')

### Analyse

**Différences v1.0 vs v2.0 :**
- v1.0 : instructions générales, l'agent décide librement
- v2.0 : protocole d'analyse de risque imposé (TOUJOURS corpus + météo + croisement), score de risque obligatoire

Le A/B testing complet sera réalisé dans le NB6 (Comparatifs).

---

---

## 7. Fallback LLM

Chaîne de fallback : Sonnet → Haiku → Mistral (Ollama local)

In [ ]:
from src.config import get_llm, get_fallback_llm, get_ollama_fallback

# Test niveau 1 : Sonnet (orchestrateur)
try:
    llm_sonnet = get_llm('orchestrator')
    print(f'  [OK] Niveau 1 — Sonnet : {llm_sonnet.model}')
except Exception as e:
    print(f'  [KO] Niveau 1 — Sonnet : {e}')

# Test niveau 2 : Haiku (fallback Anthropic)
try:
    llm_haiku = get_fallback_llm()
    print(f'  [OK] Niveau 2 — Haiku : {llm_haiku.model}')
except Exception as e:
    print(f'  [KO] Niveau 2 — Haiku : {e}')

# Test niveau 3 : Ollama (fallback local)
try:
    llm_ollama = get_ollama_fallback()
    print(f'  [OK] Niveau 3 — Ollama Mistral : disponible')
except Exception as e:
    print(f'  [INFO] Niveau 3 — Ollama : {e} (normal si Ollama non installé)')

print('>> 7. Fallback LLM : OK')

### Analyse

La chaîne de fallback garantit la disponibilité du système :
- Si l'API Anthropic est down → Haiku (moins cher, même provider)
- Si tout Anthropic est down → Mistral via Ollama (local, hors ligne, gratuit)

---

---

## 8. Dashboard MLflow

MLflow tracke automatiquement chaque question posée à l'agent. Pour visualiser :

```bash
mlflow ui --host 127.0.0.1 --port 8080
```

Puis ouvrir http://localhost:8080

In [ ]:
try:
    import mlflow
    print(f'MLflow version : {mlflow.__version__}')
    
    # Lister les expériences
    experiments = mlflow.search_experiments()
    for exp in experiments:
        print(f'  Expérience : {exp.name} (ID: {exp.experiment_id})')
    
    # Lister les derniers runs
    try:
        runs = mlflow.search_runs(experiment_names=['rag-catastrophes-climatiques'], max_results=10)
        if len(runs) > 0:
            print(f'\n  {len(runs)} runs trouvés')
            print(runs[['params.question', 'metrics.total_tokens', 'metrics.estimated_cost_usd', 'metrics.duree_s']].to_string())
        else:
            print('\n  Aucun run trouvé. Lancez d\'abord des questions via run_agent().')
    except Exception:
        print('  Expérience rag-catastrophes-climatiques non trouvée.')
        
except ImportError:
    print('MLflow non installé.')

print('>> 8. Dashboard MLflow : OK')

---

## 9. Résultats

### 9.1. Tableau synthétique

In [ ]:
# Sauvegarde
csv_path = OUTPUT_DIR / 'NB8_monitoring_results.csv'
df_tokens.to_csv(csv_path, index=False)
assert csv_path.exists(), f'Fichier non créé : {csv_path}'
print(f'  [OK] {csv_path.name} sauvegardé')
print()
print(df_tokens.to_string())

---

## 10. Conclusions

### Synthèse

[À compléter après exécution.]

---

### Quality gate finale

| Constat | Preuve | Décision pour la suite |
|---|---|---|
| Spécialisation réduit les coûts | Comparatif Haiku vs Sonnet vs Opus | Garder la spécialisation par agent |
| MLflow tracke les runs | Dashboard accessible | Utiliser pour le NB6 comparatifs |
| Fallback fonctionne | 3 niveaux testés | Production-ready |

---

### Hypothèse : [validée / invalidée]

[À compléter avec les chiffres réels.]

---

### Limites

- Le monitoring ne mesure que les tokens de l'orchestrateur, pas des sous-appels
- L'estimation de coût est basée sur les tarifs Anthropic au moment du développement
- Ollama nécessite une installation séparée

---

### Axes d'amélioration

- Monitoring des latences par outil (pas seulement par question)
- Alerting automatique si le coût dépasse un seuil
- A/B testing automatisé des prompts avec MLflow

In [ ]:
elapsed = round(time.time() - notebook_start_time, 2)
print(f'Temps total du notebook : {elapsed}s')
print('>> NOTEBOOK 8 TERMINÉ')